[SamplingBallsBinomialBootstrapping.ipynb](https://colab.research.google.com/drive/1ogFWga0l7dPigeak_JmSTCARVJFeMUco?authuser=1#scrollTo=efc5dad3)
[Drawing From Balls - Spreadsheet Result](https://docs.google.com/spreadsheets/d/1itsWZRKIU8xtnzKUfgyDgSe07tu6OL4nN1ab3s4OkPo/edit?gid=0#gid=0


In [ ]:
import random
import numpy as np
import pandas as pd

# class RelativeAndNegativeProabilities:
def generate_bag(n_total: int = 25, ball_color_counts_dict: dict=None, ball_color_prob_dict: dict=None):
    """ generates a new bag distribution """
    assert (ball_color_prob_dict is not None) or (ball_color_counts_dict is not None)
    if ball_color_prob_dict is not None:
        colors = list(ball_color_prob_dict.keys())
        probabilities = list(ball_color_prob_dict.values())

        # Generate a bag of balls based on the probabilities
        bag_of_balls = [str(x) for x in np.random.choice(colors, size=n_total, p=probabilities)]
    else:
        colors = list(ball_color_counts_dict.keys())
        counts = list(ball_color_counts_dict.values())
        bag_of_balls = random.choices(colors, weights=counts, k=n_total)

    return bag_of_balls


def sample_from_bag(bag_of_balls, n_sequential_samples: int = 25, ball_observation_error: float = 0.01):
    """ draws samples from the bag without replacement """
    bag = bag_of_balls.copy()
    samples = []
    n_samples = min(n_sequential_samples, len(bag))

    for _ in range(n_samples):
        if not bag:
            break
        idx = random.randint(0, len(bag) - 1)
        real_color = bag.pop(idx)
        # Add simulated observation error: flip color with probability = ball_observation_error
        if random.random() < ball_observation_error:
            observed_color = "white" if real_color == "orange" else "orange"
        else:
            observed_color = real_color
        samples.append(observed_color)
    return samples

_OBS_CHAR_TO_COLOR: dict[str, str] = {
    'o': 'orange',
    'w': 'white',
    'orange': 'orange',
    'white': 'white',
}

def observation_string_to_bag(obs_seq: str) -> list[str]:
    """Convert a string of white/orange observations into a full bag.

    Accepts compact chars ('o'/'w') or full color names separated by
    non-letters (e.g. 'orange,white,orange' or 'o w o').

    Usage:

        obs_seq_string_example_original_paper: str = 'owowwowwwwwwwwwwwwwwwowww'
        example_bag_from_paper = observation_string_to_bag(obs_seq_string_example_original_paper)
        display(example_bag_from_paper)
        print(pd.Series(example_bag_from_paper).value_counts().to_dict())

    """
    tokens = [t for t in ''.join(
        ch if ch.isalpha() else ' ' for ch in obs_seq.lower()
    ).split() if t]
    # Compact form with no separators: treat each character as an observation
    if len(tokens) == 1 and all(ch in _OBS_CHAR_TO_COLOR for ch in tokens[0]):
        tokens = list(tokens[0])
    try:
        return [_OBS_CHAR_TO_COLOR[t] for t in tokens]
    except KeyError as exc:
        raise ValueError(
            f"Unknown observation token {exc.args[0]!r}; expected o/w or orange/white"
        ) from exc

n_total: int = 25

# Generate a random bag
ball_color_prob_dict = {'orange': 3.0/float(n_total), 'white': (n_total - 3.0)/ float(n_total)}
generated_bag = generate_bag(n_total=n_total, ball_color_prob_dict=ball_color_prob_dict)
display(generated_bag)
samples = sample_from_bag(bag_of_balls=generated_bag)
display(samples)

# Generate a fixed bag
ball_color_counts_dict = {'orange': 3, 'white': (n_total - 3)}
generated_bag = generate_bag(n_total=n_total, ball_color_counts_dict=ball_color_counts_dict)
display(generated_bag)


['white',
 'white',
 'white',
 'white',
 'orange',
 'white',
 'orange',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white']

['white',
 'white',
 'orange',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'orange',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white']

['white',
 'orange',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'orange',
 'orange',
 'white',
 'white',
 'white']

In [ ]:
obs_seq_string_example_original_paper: str = 'owowwowwwwwwwwwwwwwwwowww'
example_bag_from_paper = observation_string_to_bag(obs_seq_string_example_original_paper)
display(example_bag_from_paper)
print(pd.Series(example_bag_from_paper).value_counts().to_dict())


['orange',
 'white',
 'orange',
 'white',
 'white',
 'orange',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'white',
 'orange',
 'white',
 'white',
 'white']

{'white': 21, 'orange': 4}


In [2]:
measurement_noise_probabilities: pd.DataFrame = pd.DataFrame([[0.99, 0.01], [0.01, 0.99]], columns=['real_white', 'real_orange'], index=['obs_white', 'obs_orange'])
measurement_noise_probabilities

,real_white,real_orange
obs_white,0.99,0.01
obs_orange,0.01,0.99


In [3]:
def observe_ball(real_color: str, noise_df: pd.DataFrame):
    """
    Simulates observing a ball with measurement noise.
    """
    # Map the real color name to the column name in the noise dataframe
    column_name = f'real_{real_color}'

    # Get the possible observations and their probabilities
    observations = noise_df.index.tolist()
    probabilities = noise_df[column_name].values

    # Randomly choose an observation
    observation = np.random.choice(observations, p=probabilities)

    # Strip the 'obs_' prefix to return just the color
    return observation.replace('obs_', '')

# Demonstrate observing the entire bag
observed_bag = [observe_ball(ball, measurement_noise_probabilities) for ball in generated_bag]

print(f"Real bag (first 10): {generated_bag[:10]}")
print(f"Observed bag (first 10): {observed_bag[:10]}")

# Compare counts
print("\nReal Counts:", pd.Series(generated_bag).value_counts().to_dict())
print("Observed Counts:", pd.Series(observed_bag).value_counts().to_dict())

Real bag (first 10): ['white', 'orange', 'white', 'white', 'white', 'white', 'white', 'white', 'white', 'white']
Observed bag (first 10): ['white', 'orange', 'white', 'white', 'white', 'white', 'white', 'white', 'white', 'white']

Real Counts: {'white': 22, 'orange': 3}
Observed Counts: {'white': 21, 'orange': 4}


In [4]:
Implement an observation event, which consumes a ball in a bag and returns the color of the ball with a certain correctness probability, simulating measurement error.
The user defines a table `measurement_noise_probabilities: pd.DataFrame = pd.DataFrame([[0.99, 0.01], [0.01, 0.99]], columns=['real_white', 'real_orange'], index=['obs_white', 'obs_orange'])` which gives the probabilities for observing a color when the drawn color was really a color.

SyntaxError: invalid syntax (3552608109.py, line 1)